# WarpAlign CUDA validation
Select a GPU runtime first. This notebook validates experimental code; it does not establish a speedup. Use synthetic/public data only. Save logs before the Colab session ends.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)


In [ ]:
# First run locally: python scripts/package_colab.py
# Upload /tmp/warpalign-colab.zip; no commit or push is required.
from google.colab import files
from pathlib import Path
import os, tempfile, zipfile, subprocess
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one WarpAlign source ZIP"
archive = Path(next(iter(uploaded))).resolve()
root = Path(tempfile.mkdtemp(prefix="warpalign-", dir="/content")).resolve()
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (root / name).resolve().is_relative_to(root), "Unsafe archive path"
    z.extractall(root)
os.chdir(root)
subprocess.run(["python", "-m", "pip", "install", "-e", ".[test]", "cmake", "pybind11"], check=True)


The source archive includes uncommitted changes. The validation script must pass compilation, actual device scoring/traceback, memory checks and Python dispatch. A missing Compute Sanitizer is a prerequisite failure, not a passed GPU test.


In [ ]:
import subprocess
subprocess.run(["bash", "scripts/validate_cuda.sh"], check=True)


In [ ]:
import os
os.environ["WARPALIGN_EXPERIMENTAL_CUDA"]="1"
import warpalign
a=warpalign.Aligner(backend="cuda",allow_fallback=False)
r=a.align("ACGT", "ACGTT")
print(r.score,r.cigar,r.backend)
assert r.backend=="cuda" and r.score==8 and r.cigar=="4=1D"


In [ ]:
subprocess.run(["python", "benchmarks/scripts/run.py", "--lengths", "100,1000,10000", "--batches", "1,16", "--backend", "cuda", "--no-fallback", "--output", "benchmarks/results/colab"], check=True)
